# Atlas differential expression

Load the Harmony atlas from [`run_atlas_harmony.py`](../../pipelines/run_atlas_harmony.py). Full-gene counts live in `.raw`; `.X` holds normalized HVGs for embedding. Disease areas come from [`disease_markers.labels`](../../scripts/disease_markers/labels.py).

In [ ]:
from pathlib import Path

import decoupler as dc
import numpy as np
import pandas as pd
import scanpy as sc
from disease_markers.labels import build_sample_label_table
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from shared.repo import REPO_ROOT

sc.set_figure_params(figsize=(3, 3), frameon=False)

REPO = REPO_ROOT
# HARMONY_H5AD = REPO / "output/atlas/v1/processed_1/atlas_harmony_raw_sub_5k.h5ad"
HARMONY_H5AD = REPO / "output/atlas/v1/processed_2_raw/atlas_harmony_raw.h5ad"
CONTEXTS = REPO / "output/context/contexts.jsonl"
ATLAS_CSV = REPO / "output/atlas/v1/atlas.csv"
SAMPLE_KEY = "SRX_accession"
STUDY_KEY = "study_accession"
CLUSTER_KEY = "leiden_atlas"
MIN_CELLS_PER_PROFILE = 10

In [ ]:
harmony = sc.read_h5ad(HARMONY_H5AD)
if harmony.raw is None:
    msg = "Expected full-gene counts in adata.raw; re-run run_atlas_harmony on an atlas built after the raw-count change"
    raise ValueError(msg)

print(f"Harmony object: {harmony.n_obs:,} cells x {harmony.n_vars:,} HVGs; raw {harmony.raw.n_vars:,} genes")
harmony.obs.columns

In [ ]:
harmony

In [ ]:
import numpy as np
rng = np.random.default_rng(0)


In [ ]:
atlas = harmony.raw.to_adata()
for col in harmony.obs.columns:
    atlas.obs[col] = harmony.obs[col].values
for key in harmony.obsm.keys():
    atlas.obsm[key] = harmony.obsm[key]

assert atlas.n_vars == harmony.raw.n_vars
atlas.n_vars, atlas.X.sum()

In [ ]:
label_table = build_sample_label_table(CONTEXTS, ATLAS_CSV)
labels_by_srx = label_table.set_index("srxAccession")

atlas.obs["diseaseArea"] = atlas.obs[SAMPLE_KEY].map(labels_by_srx["diseaseArea"]).astype("category")
atlas.obs["isControl"] = atlas.obs[SAMPLE_KEY].map(labels_by_srx["isControl"]).fillna(False)
eligible = atlas.obs[SAMPLE_KEY].map(labels_by_srx["eligible"]).fillna(False)
atlas = atlas[eligible.to_numpy()].copy()

label_table.head(), atlas.obs["diseaseArea"].value_counts()

In [ ]:
atlas.obs["diseaseArea"].value_counts(normalize=True)

In [ ]:
import pandas as pd
from shared.repo import REPO_ROOT
metadata = pd.read_parquet(
    REPO_ROOT
    / "data/scbasecount/2026-01-12/metadata/GeneFull/Homo_sapiens/scbasecount_2026-01-12_metadata_GeneFull_Homo_sapiens_sample_metadata.parquet"
)
metadata.set_index("srx_accession", inplace=True)
scbasecount_disease = metadata["disease"]
atlas.obs["disease"] = atlas.obs[SAMPLE_KEY].map(scbasecount_disease)
atlas.obs["disease"].value_counts(normalize=True).plot.bar()

In [ ]:
# sc.set_figure_params(figsize=(5, 5), frameon=False)
sc.pl.umap(atlas, color="diseaseArea", show=False).figure.savefig(Path().resolve().parents[1] / "tmp/atlas_disease_area.png", dpi=300)

In [ ]:
# sc.pl.embedding(harmony, basis="X_umap", color=[CLUSTER_KEY, STUDY_KEY])
harmony.obs

In [ ]:
pb_result = dc.pp.pseudobulk(
    atlas,
    sample_col=SAMPLE_KEY,
    groups_col=CLUSTER_KEY,
    mode="sum",
)
pdata = pb_result[0] if isinstance(pb_result, tuple) else pb_result
# pdata = pdata[pdata.obs["psbulk_cells"] >= MIN_CELLS_PER_PROFILE].copy()

area_for_sample = labels_by_srx["diseaseArea"]
pdata.obs["diseaseArea"] = pdata.obs[SAMPLE_KEY].map(area_for_sample).astype("category")
pdata.obs[STUDY_KEY] = pdata.obs[SAMPLE_KEY].map(labels_by_srx["studyAccession"])
pdata.obs.head()

## One cluster, one-vs-rest DESeq2

`de_all_cluster_area_de` runs every Leiden cluster against every disease area (except Control). It returns `de_summary`, filtered `de_hits`, and full `de_results` (all genes, for volcanoes).

In [ ]:
def counts_dataframe(adata: sc.AnnData) -> pd.DataFrame:
    matrix = adata.X
    if hasattr(matrix, "toarray"):
        matrix = matrix.toarray()
    counts = np.rint(np.asarray(matrix, dtype=np.float64)).astype(int)
    return pd.DataFrame(counts, index=adata.obs_names, columns=adata.var_names)


def one_vs_rest_de_full(pdata: sc.AnnData, area: str) -> pd.DataFrame:
    meta = pdata.obs.copy()
    meta["group"] = np.where(meta["diseaseArea"].astype(str) == area, area, "rest")
    if meta["group"].nunique() < 2 or (meta["group"] == area).sum() < 2:
        return pd.DataFrame()
    counts = counts_dataframe(pdata)
    dds = DeseqDataSet(
        counts=counts,
        metadata=meta,
        design_factors="group",
        ref_level=["group", "rest"],
        quiet=True,
    )
    dds.deseq2()
    stats = DeseqStats(dds, contrast=["group", area, "rest"], quiet=True)
    stats.summary()
    results = stats.results_df.copy()
    results["gene"] = results.index.astype(str)
    return results.reset_index(drop=True)


def one_vs_rest_de(
    pdata: sc.AnnData, area: str, *, padj: float = 0.05, lfc: float = 1.0
) -> pd.DataFrame:
    results = one_vs_rest_de_full(pdata, area)
    if results.empty:
        return results
    return results[
        (results["padj"] <= padj)
        & (results["log2FoldChange"] >= lfc)
        & results["padj"].notna()
    ].reset_index(drop=True)


def de_all_cluster_area_de(
    pdata: sc.AnnData,
    *,
    padj: float = 0.05,
    lfc: float = 1.0,
    skip_areas: frozenset[str] = frozenset({"Control"}),
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Run one-vs-rest DESeq2 for every Leiden cluster and disease area.

    Returns (summary, hits, results). results holds all genes per cluster x area
    (for volcano plots); hits is the padj/lfc-filtered long table.
    """
    clusters = sorted(pdata.obs[CLUSTER_KEY].astype(str).unique(), key=str)
    areas = sorted(
        a for a in pdata.obs["diseaseArea"].astype(str).unique() if a not in skip_areas
    )
    summary_rows: list[dict[str, object]] = []
    hit_frames: list[pd.DataFrame] = []
    result_frames: list[pd.DataFrame] = []

    for cluster in clusters:
        cluster_pdata = pdata[pdata.obs[CLUSTER_KEY].astype(str) == cluster].copy()
        n_pseudobulks = cluster_pdata.n_obs
        for area in areas:
            n_area = int((cluster_pdata.obs["diseaseArea"].astype(str) == area).sum())
            full = one_vs_rest_de_full(cluster_pdata, area)
            if full.empty:
                hits = full
            else:
                hits = full[
                    (full["padj"] <= padj)
                    & (full["log2FoldChange"] >= lfc)
                    & full["padj"].notna()
                ].reset_index(drop=True)
            summary_rows.append(
                {
                    "cluster": cluster,
                    "diseaseArea": area,
                    "nPseudobulks": n_pseudobulks,
                    "nAreaPseudobulks": n_area,
                    "nHits": len(hits),
                }
            )
            if full.empty:
                continue
            tagged_full = full.copy()
            tagged_full.insert(0, "cluster", cluster)
            tagged_full.insert(1, "diseaseArea", area)
            result_frames.append(tagged_full)
            if hits.empty:
                continue
            tagged_hits = hits.copy()
            tagged_hits.insert(0, "cluster", cluster)
            tagged_hits.insert(1, "diseaseArea", area)
            tagged_hits["nPseudobulks"] = n_pseudobulks
            tagged_hits["nAreaPseudobulks"] = n_area
            hit_frames.append(tagged_hits)

    summary = pd.DataFrame(summary_rows)
    hits = pd.concat(hit_frames, ignore_index=True) if hit_frames else pd.DataFrame()
    empty_results = pd.DataFrame(
        columns=[
            "cluster",
            "diseaseArea",
            "gene",
            "baseMean",
            "log2FoldChange",
            "lfcSE",
            "stat",
            "pvalue",
            "padj",
        ]
    )
    results = pd.concat(result_frames, ignore_index=True) if result_frames else empty_results
    return summary, hits, results


de_summary, de_hits, de_results = de_all_cluster_area_de(pdata)
de_summary

In [ ]:
import matplotlib.pyplot as plt


def _volcano_on_axes(
    ax: plt.Axes,
    results: pd.DataFrame,
    title: str,
    *,
    padj_threshold: float = 0.05,
    lfc_threshold: float = 1.0,
) -> None:
    if results.empty:
        ax.set_title(f"{title}\n(skipped)", fontsize=8)
        ax.axis("off")
        return

    df = results.copy()
    padj = df["padj"].fillna(1.0).clip(lower=1e-300)
    df["neglog10Padj"] = -np.log10(padj)
    sig = (df["padj"] <= padj_threshold) & (df["log2FoldChange"].abs() >= lfc_threshold)

    ax.scatter(
        df.loc[~sig, "log2FoldChange"],
        df.loc[~sig, "neglog10Padj"],
        s=3,
        c="lightgray",
        alpha=0.5,
        rasterized=True,
    )
    ax.scatter(
        df.loc[sig, "log2FoldChange"],
        df.loc[sig, "neglog10Padj"],
        s=5,
        c="crimson",
        alpha=0.75,
        rasterized=True,
    )
    ax.axvline(lfc_threshold, color="0.4", ls="--", lw=0.7)
    ax.axvline(-lfc_threshold, color="0.4", ls="--", lw=0.7)
    ax.axhline(-np.log10(padj_threshold), color="0.4", ls="--", lw=0.7)
    ax.set_title(title, fontsize=8)
    ax.set_xlabel("log2 FC")
    ax.set_ylabel(r"$- \log_{10}$ padj")


def plot_volcanoes_by_cluster(
    de_results: pd.DataFrame,
    de_summary: pd.DataFrame,
    *,
    padj_threshold: float = 0.05,
    lfc_threshold: float = 1.0,
    max_cols: int = 3,
) -> None:
    """One figure per cluster; one volcano subplot per disease area."""
    if de_summary.empty:
        return

    has_gene_results = "cluster" in de_results.columns and "diseaseArea" in de_results.columns

    clusters = sorted(de_summary["cluster"].astype(str).unique(), key=str)
    for cluster in clusters:
        area_rows = de_summary[de_summary["cluster"].astype(str) == cluster]
        areas = area_rows["diseaseArea"].astype(str).tolist()
        n = len(areas)
        ncols = min(max_cols, n)
        nrows = int(np.ceil(n / ncols))
        fig, axes = plt.subplots(
            nrows,
            ncols,
            figsize=(3.4 * ncols, 3.2 * nrows),
            squeeze=False,
        )
        for i, area in enumerate(areas):
            ax = axes[i // ncols][i % ncols]
            if has_gene_results:
                mask = (de_results["cluster"].astype(str) == cluster) & (
                    de_results["diseaseArea"].astype(str) == area
                )
                area_results = de_results.loc[mask]
            else:
                area_results = pd.DataFrame()
            _volcano_on_axes(
                ax,
                area_results,
                area,
                padj_threshold=padj_threshold,
                lfc_threshold=lfc_threshold,
            )
        for j in range(n, nrows * ncols):
            axes[j // ncols][j % ncols].axis("off")
        fig.suptitle(f"Cluster {cluster}", fontsize=11, y=1.02)
        fig.tight_layout()
        plt.show()


plot_volcanoes_by_cluster(de_results, de_summary)